# 42 새 질문 48개로 검색 방식 확대 평가
운영 코드·OCR·이전 결과는 읽기 전용입니다. 실험은 순차 실행합니다.
이 노트북은 입력 → 실행 상태 → 평가 → 비용을 저장 결과로 확인합니다. 유료 호출은 자동 실행하지 않습니다.
장시간 실행·원응답 journaling·동일 payload 캐시·메모리 중단/재개가 필요해 `42_expanded_query_evaluation.py`를 별도 runner로 사용합니다.
재현: 기존 승인 환경에서 `python -B notebooks/42_expanded_query_evaluation.py` 실행 후 이 노트북의 전체 표시 셀을 실행합니다.
모델·자료·평가 기준은 protocol.json과 소스 해시로 고정합니다. 같은 질문 재호출 분산은 측정하지 않습니다.


In [ ]:
from pathlib import Path
import json
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
DATA = ROOT / "notebooks/data/42_expanded_query_evaluation"
def read(name): return json.loads((DATA / name).read_text())
protocol = read("protocol.json")
print(json.dumps(protocol, ensure_ascii=False, indent=2)[:10000])


## 실행 결과
검색·답변 실패도 분모에 포함합니다. 일반/모호/자료 미확인 질문은 양성 지표에서 분리합니다.


In [ ]:
print(json.dumps(read("run_summary.json"), ensure_ascii=False, indent=2))


## 평가
Hit@3는 상위3개 정답 카드 포함, MRR은 첫 정답 순위 역수 평균, 답변충족은 원문 상대 자동 조건 검사의 전체 통과입니다. 높을수록 좋습니다.
카드가 검색돼도 혜택·조건을 답변에서 빠뜨리면 답변충족에 실패할 수 있습니다. 문자열·동의어·부정문 오탐과 불완전한 원문 관계를 원응답으로 따로 점검합니다.
benefit과 parent-child는 청킹 외 재정렬 프로필에도 차이가 있습니다.


In [ ]:
import pandas as pd
evaluation = read("evaluation_summary.json")
display(pd.DataFrame(evaluation["metrics"]))
print(json.dumps(evaluation["paired"], ensure_ascii=False, indent=2))


## 비용·원본 보존
API 실제 물리 요청과 캐시 재사용을 분리합니다. 사용량×기록 단가 환산이며 실제 청구액은 미측정입니다. 시간·CPU·peak RSS는 해당 프로세스의 측정값입니다.


In [ ]:
print(json.dumps(read("cost_accounting.json")["summary"], ensure_ascii=False, indent=2))
print(json.dumps(read("preservation.json"), ensure_ascii=False, indent=2))
